# 01b — Scrape ccb.bh (Constitutional Court)

**Status: automated pull works — 93/93 rulings recovered in ~2.3 minutes, zero failures.**

- There are **93 rulings** total (an earlier "22 rulings" estimate was wrong). Case ID formats are
  inconsistent (`ح/2019/2`, `د/1/2014`, `ط-ح-1-2011`, `م ت1-05`, etc.) — these are Constitutional Court
  reference numbers, not something to parse; the raw text is kept as-is for citation.
- **All 93 rulings are present in the page's DOM on load**, just CSS-collapsed under year headers.
  Selenium's `.text` property is visibility-aware and returns `""` for collapsed elements — use
  `get_attribute("textContent")` instead, which ignores visibility.
- Each ruling triggers a classic ASP.NET WebForms `__doPostBack`, which returns a **PDF** directly.
  Replicated via `requests.Session().post()` using the page's live `__VIEWSTATE`/`__EVENTVALIDATION`.
- **An earlier attempt hit a redirect loop after 3 successful downloads and looked like a hard wall —
  it wasn't.** Retrying with fresh page state on failure (reload the list page, re-read VIEWSTATE, keep
  going) got all 93/93 cleanly on the very next attempt. Treat this site the same as sjc.bh/lloc.gov.bh:
  transient failures under sustained requests are normal for these government sites, not permanent
  blocks — always retry before concluding something is broken.
- Confirmed earlier: PDFs are **digital-native, not scanned** (embedded font objects, no image/OCR
  filters) — clean text extraction via PyMuPDF, no OCR required.
- PDFs are saved to `data/raw/ccb/pdfs/` (named after their case ID) as well as having their text
  extracted into `ccb_rulings.json`, so the raw files are kept even if the JSON is regenerated later.


In [1]:
import time
import re
import json
from pathlib import Path

import requests
from selenium import webdriver
from selenium.webdriver.common.by import By
import fitz  # PyMuPDF

BASE = "https://www.ccb.bh"
LIST_URL = BASE + "/Pages_ar/Listdoc.aspx?encr=1B3A&mtype=anVk"
PDF_DIR = Path("../data/raw/ccb/pdfs")
PDF_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR = Path("../data/raw/ccb")


### Step 1 — open a Selenium session, load the list page, extract all 93 rulings

In [2]:
driver = webdriver.Chrome()
driver.get(LIST_URL)
time.sleep(5)

links = driver.find_elements(By.CSS_SELECTOR, "a[href*='__doPostBack']")
rulings = []
for a in links:
    # a.text is visibility-aware and returns "" for CSS-collapsed (accordion) links — use
    # textContent instead, which ignores visibility.
    text = (a.get_attribute("textContent") or "").strip()
    if not text or not re.search(r"\d", text):
        continue  # excludes the 2 non-ruling nav links (no digit in their text at all)
    href = a.get_attribute("href")
    m = re.search(r"__doPostBack\('([^']+)','([^']*)'\)", href)
    if m:
        rulings.append({"case_id": text, "event_target": m.group(1), "event_argument": m.group(2)})

print(f"Found {len(rulings)} rulings (expect 93)")

cookies = driver.get_cookies()
session = requests.Session()
for c in cookies:
    session.cookies.set(c["name"], c["value"])
session.headers.update({"User-Agent": driver.execute_script("return navigator.userAgent")})


Found 93 rulings (expect 93)


### Step 2 — replicate each postback, with retry-on-failure (reload page, refresh state)

In [3]:
def get_form_state():
    fields = {}
    for name in ("__VIEWSTATE", "__VIEWSTATEGENERATOR", "__EVENTVALIDATION"):
        try:
            el = driver.find_element(By.ID, name)
            fields[name] = el.get_attribute("value")
        except Exception:
            fields[name] = ""
    return fields


def fetch_ruling_pdf(ruling: dict, retries: int = 3) -> bytes:
    """Replicates the __doPostBack as a requests POST. Retries by reloading the list page and
    re-reading VIEWSTATE on failure — confirmed live that a redirect loop after a few successful
    requests is a transient site issue, not a permanent block (0 failures across 93 rulings once
    retry-with-fresh-state was added).
    """
    last_exc = None
    for attempt in range(retries):
        try:
            form_state = get_form_state()
            payload = {**form_state, "__EVENTTARGET": ruling["event_target"], "__EVENTARGUMENT": ruling["event_argument"]}
            r = session.post(LIST_URL, data=payload, timeout=30)
            r.raise_for_status()
            if r.headers.get("Content-Type", "").startswith("application/pdf"):
                return r.content
            raise ValueError(f"unexpected content-type: {r.headers.get('Content-Type')}")
        except Exception as e:
            last_exc = e
            if attempt < retries - 1:
                driver.get(LIST_URL)
                time.sleep(3)
                for c in driver.get_cookies():
                    session.cookies.set(c["name"], c["value"])
                time.sleep(1)
    raise last_exc


# Smoke test on one ruling
if rulings:
    pdf_bytes = fetch_ruling_pdf(rulings[0])
    print(f"{rulings[0]['case_id']}: {len(pdf_bytes)} bytes, starts with {pdf_bytes[:8]}")


م.ت/2025/1: 2731772 bytes, starts with b'%PDF-1.4'


### Step 3 — pull all 93, save PDFs to disk, extract text (digital-native, no OCR needed)

In [4]:
def pull_all_ccb(rulings, pdf_dir: Path, out_dir: Path, delay: float = 0.8):
    records = []
    for i, ruling in enumerate(rulings):
        try:
            pdf_bytes = fetch_ruling_pdf(ruling)
            safe_name = ruling["case_id"].replace("/", "_").replace(" ", "_")
            (pdf_dir / f"{safe_name}.pdf").write_bytes(pdf_bytes)
            doc = fitz.open(stream=pdf_bytes, filetype="pdf")
            text = "\n".join(page.get_text() for page in doc)
            records.append({**ruling, "text": text, "pages": len(doc)})
            print(f"OK  [{i+1}/{len(rulings)}] {ruling['case_id']}: {len(doc)} pages")
        except Exception as e:
            print(f"FAIL [{i+1}/{len(rulings)}] {ruling['case_id']}: {e}")
        time.sleep(delay)

    out_path = out_dir / "ccb_rulings.json"
    out_path.write_text(json.dumps(records, ensure_ascii=False, indent=1), encoding="utf-8")
    print(f"\nSaved {len(records)}/{len(rulings)} rulings -> {out_path}")
    return records


records = pull_all_ccb(rulings, PDF_DIR, OUT_DIR)


OK  [1/93] م.ت/2025/1: 17 pages


OK  [2/93] م.ت/2023/1: 10 pages


OK  [3/93] م.ت/2021/2: 13 pages


OK  [4/93] م.ت/2022/1: 10 pages


OK  [5/93] م.ت/2021/1: 9 pages


...

OK  [90/93] د/03/3: 12 pages


OK  [91/93] د/04/1: 3 pages


OK  [92/93] د/03/1: 8 pages


OK  [93/93] د/03/2: 11 pages



Saved 93/93 rulings -> ..\data\raw\ccb\ccb_rulings.json
